#  Mini-Project 2: Travelling between Inner and Outer London, Peak or Off-Peak Hours

**DS105W Mini-Project 2, Data for Data Science (Winter Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Student Notebook**
- 📅 Date: 1 Apr 2026 (due date)
- 👤 Name: Mingyun Wang
- 📛 Candidate Number: 70643
- 🎯 Purpose of this Notebook: Collecting Data


</div>

**NOTE: DO NOT CLICK ON THE 'RUN ALL' BUTTON TO RUN THIS NOTEBOOK BEFORE YOU MODIFY THE INPUT VARIABLES FOR DATA REQUESTS!!!**

In this notebook, data requests were sent for 18 Mar 2026, and requests sent on any date after that day will produce undesired output, because the TfL API is designed for future journey plannings.

***For demonstration versions (not directly downloaded from Github), you don't need to worry because I will have changed the inputs for your convenience. The input time of my original version was 19 Mar 2026 (Thu). Demonstration versions will feature 21 Mar 2030 (Thu).
However, the expected time lengths to run the data collection cells will not be reliable for and demonstration vversions.***

## The Overall Plan is here 👇:
You probably have been able to access this overall plan via README.md

- the Inner London area is **LSE Main Campus** (as a default set by the mini project requirement, and controlling this as a single destination helps derive simplicity and standardisation over a range of omitted variables), and the Outer London boroughs: **Ealing, Sutton, Barking, and Enfield**. Reasons why I chose these are briefed in NB01.
- Some **sample postcodes** will be selected for API requests regarding journey plans based on their LSOA directories and usertypes. (*See NB01 for how those postcodes are chosen*)

- there are a few types of the investigated variables: *(note this will also appear in NB02)*
  1. `Identity Information:`
    - **postcode** (of specific spots of LSOAs subject to boroughs)
    - **timeband** (peak or offpeak)
    - **borough** (one of Ealing, Sutton, Barking, or Enfield)
    - **usertype** (binary, 0 for residential, 1 for business)
    - **IMD** (index, how deprive an LSOA is)
  2. `Time-Related Information:`
    - **duration** (averaged for inward and outword journeys)
    - **walking minutes** (averaged)
    - **transport minutes** (averaged, non-walking time during a journey, or AKA time spent on transportations)
  3. `Mode-Related Information:`
    - **complexity** (averaged, the number of transfers/transits/switches of travelling mode, obtainable by counting the number of a journey's legs)
    - **major mode** (the non-walking rode that one is estimated to spend the most time on during a journey)
  4. `Cost Information:`
    - **fare total cost** (averaged)
  - `NOTE` that several variables, as marked, are the **average** of a sample journey from outer London to inner London and the average of a sample journey form inner London to outer London. This is to balance the subtle difference between travelling inwards and outwards due to a range of reasons, like the convenience of getting to bus stops and underground stations.

- **Peak and Off-Peak Hours**: (*for more detailed definition and reason, see NB01*)
  - for **Peak Hours**, data are collected for **either 2026 Mar 19 (Thu) 08:00 or 2026 Mar 19 (Thu) 17:00**
  - for **Off-Peak Hours**, data are collected for **only 2026 Mar 19 (Thu) 12:00**

- There is a **Trial** Workflow and an **Extension** Workflow for NB01 and NB02:
  - This arrangement is because I would like to write my Data Collection and Transformation workflows with Ealing into functions, so that the amount of work will be reduced for more boroughs in terms of these repetitive jobs.
  - The **Trial** workflow picks one outer london area and one inner london area only, specifically the inner London spot will be **LSE Main Campus** (as the project's default) and the outer London borough will be **Ealing** (not a single spot, including the districts of Acton, Ealing, Greenford, Hanwell, Northolt, Perivale and Southall)
  - These choices pretty make sense because both places are Bustling Busy Business areas surrounded by/involving a wide range of businesses and residential spots. Then it would be quite reasonable to expect that we could see more intense commuting flows from A to B and a clearer difference between peak/off-peak hours.
  - The **Extension** workflow is expected to engage the rest of the boroughs: Sutton, Barking, and Enfield.
  - I complete a whole cycle of the Trial, and only then will I work with the Extension
- NB03 pools data together for analyses

## 1.0, Planning this Notebook

### 1.0.1, In This Notebook 📓:
1. I will **select some postcodes** from each borough as samples:
  - these postcodes must be active in use
  - for each borough, sample postcodes must come from a mix of the two usertypes: residential (0) or business (1) for generality
  - up to **3 postcodes will be selected for each LSOA**, for the simplicity and request volume concerns
2. I will **define my Peak and Offpeak Hours** for later requests and analyses
2. For each postcodes, there will be **4 requests** sent to the TfL API:
  - travelling from the spot to LSE (inward travel) during peak hours
  - travelling from LSE to the spot (outward travel) during peak hours
  - travelling from the spot to LSE (inward) during offpeak hours
  - travelling from the LSE to the spot (outward) during offpeak hours
  - *P.S.averages with respect to each postcode and timaband (peak/offpeak) will be taken in NB02*
3. All data in this notebook will be **saved into the '../data/raw/'** (this is the relative path from any of the notebooks) **directory**

### 1.0.2, Necessary Definitions before I have a Lift Off
1. What are Peak/Off-Peak Hours?
   By TfL's definition, **Peak Hours are 7:30-9:00 am and 5:00-7:00 pm, Mon to Fri**, and naturally the rest are off-peak hours from Mon to Fri.

2. How do I sample for Peak/Off-Peak Hours?
   Considering that my samples of postcodes in each borough covers a range of spots, and travelling inwards and outwards count differently during data collection, I'm afraid requesting for data for multiple time points within the peak/off-peak definitions will make my VSCode explode. Then I pick some representative samples of peak/off-peak hours while collecting data:
   - for **Peak Hours**, data are collected for **either 2026 Mar 19 (Thu) 08:00 or 2026 Mar 19 (Thu) 17:00**
   - for **Off-Peak Hours**, data are collected for **only 2026 Mar 19 (Thu) 12:00**

3. The reasons for selecting these samples for peak/off-peak hours:
   - considering the **9-5 nature of many jobs**, I expect commuting behaviours between outer and inner London to reach their peaks at eight in the morning (when poor workers/students rush to their jobs/lectures) and at five in the evening (when we call it a day and wrap up our works).
   - 12pm during weekdays is observably an ideal off-peak sample, for I've taken London undergrounds around this time multiple times and I simply know that.
   - **Mar 19 is simply a typical Thursday for London**, there is practically **nothing really special** on the day, nor will there be closures of major underground lines. Thus I expect there should not be significant influence because of extraneous factors to people's travelling/commuting patterns.

## 1.1, Simplifying the ONS London Postcodes Table

### 1.1.0, Libraries

In [2]:
import os
%pip install requests
%pip install dotenv
import requests
from dotenv import load_dotenv
import csv
import json
import pandas as pd
import numpy as np
import time

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### 1.1.1, Tidying Up the ONS Table

In [3]:
# the ONS table lies in the "mini-project-2/data/raw" directory
ons = pd.read_csv('../data/raw/london_postcodes-ons-postcodes-directory-feb22.csv')
display(ons.head(3))
len(ons)

,pcd,pcd2,pcds,dointr,doterm,oscty,ced,oslaua,osward,parish,...,ru11ind,oac11,lat,long,lep1,lep2,pfa,imd,calncv,stp
0,BR1 1AA,BR1 1AA,BR1 1AA,201605,NaN,E99999999,E99999999,E09000006,E05000109,E43000196,...,A1,4C3,51.401546,0.015415,E37000051,NaN,E23000001,24305,E56000010,E54000030
1,BR1 1AB,BR1 1AB,BR1 1AB,201203,NaN,E99999999,E99999999,E09000006,E05000109,E43000196,...,A1,2D1,51.406333,0.015208,E37000051,NaN,E23000001,13716,E56000010,E54000030
2,BR1 1AD,BR1 1AD,BR1 1AD,201409,201709.0,E99999999,E99999999,E09000006,E05000109,E43000196,...,A1,4C3,51.400057,0.016715,E37000051,NaN,E23000001,24305,E56000010,E54000030


326214

Some postcodes have been terminated from use, featuring a non-empty "doterm" (date of termination) value. Then select those active postcodes with empty "doterm" entries (and naturally say bye-bye to those useless rows). As the output comes out we can see that roughly 40% of the rows are useless.

In [4]:
ons = ons[ons['doterm'].isna()].reset_index()
ons = ons.drop(columns='index')
display(ons.head(3))
len(ons)

,pcd,pcd2,pcds,dointr,doterm,oscty,ced,oslaua,osward,parish,...,ru11ind,oac11,lat,long,lep1,lep2,pfa,imd,calncv,stp
0,BR1 1AA,BR1 1AA,BR1 1AA,201605,NaN,E99999999,E99999999,E09000006,E05000109,E43000196,...,A1,4C3,51.401546,0.015415,E37000051,NaN,E23000001,24305,E56000010,E54000030
1,BR1 1AB,BR1 1AB,BR1 1AB,201203,NaN,E99999999,E99999999,E09000006,E05000109,E43000196,...,A1,2D1,51.406333,0.015208,E37000051,NaN,E23000001,13716,E56000010,E54000030
2,BR1 1AE,BR1 1AE,BR1 1AE,200808,NaN,E99999999,E99999999,E09000006,E05000109,E43000196,...,A1,2D1,51.404543,0.014195,E37000051,NaN,E23000001,20694,E56000010,E54000030


180126

**Keeping ONLY NECESSARY, "Core" Columns**

Here, I copied Jon's steps in Week07 NB02 directly. The old, comprehensive ONS table is preserved as "ons", and a new "ons_core" is made as a copy of necessary columns from the old table.

In [5]:
expected_columns = [
    "oslaua",
    "pcds",
    "msoa11",
    "lsoa11",
    "usertype",
    "lat",
    "long",
    "imd"
]

core_columns = [
    col
    for col in expected_columns
    if col in ons.columns
]

ons_core = ons[core_columns].copy()    # creating a core ons table and preserving the comprehensive ons table
display(ons_core.head(3))
len(ons_core)

,oslaua,pcds,msoa11,lsoa11,usertype,lat,long,imd
0,E09000006,BR1 1AA,E02000144,E01000675,0,51.401546,0.015415,24305
1,E09000006,BR1 1AB,E02000134,E01000676,0,51.406333,0.015208,13716
2,E09000006,BR1 1AE,E02000144,E01000677,0,51.404543,0.014195,20694


180126

I feel like I'd better save this to a csv file in the "project/data/processed" directory

In [6]:
ons_core.to_csv('../data/processed/ons-core.csv', index=False)

## 1.2, TRIAL: London Borough of Ealing
1. Selecting Postcodes for Ealing
2. Requesting Function for Data Collection and Storage

Of course functions in this section came from trials and errors with Ealing's case. But since both functions are pretty simple in their principles, I'd directly put and briefly explain the functions without showing the process of manual trials.

### 1.2.1, Selecting Postcodes for Boroughs

I'm defining here **A Function to Fetch ONS Information for A Certain Borough** from the "ons_core" table. It works by pairing borough with its code and adds an additional column that **tags** the borough name in the end of the borough's table -- I feel like this should be useful later on.

Selected dataframe for each borough's ONS information will be saved to the /data/raw directory as .csv files

In [7]:
def ons_borough(borough, code):
    ons_borough = ons_core[ons_core['oslaua'] == code]
    ons_borough['borough'] = np.where(ons_borough['oslaua'] == code, borough, '')
    ons_borough = ons_borough.dropna()    # to ensure there won't be any empty values in the resultant table
    ons_borough = ons_borough.groupby(['usertype']).head(30)
    ons_borough = ons_borough.groupby(['lsoa11']).head(3)    # to reduce the number of postcodes within each LSOA; reduce request burdens later
    ons_borough.to_csv(f'../data/raw/ons_{borough}.csv', index=False)
    return ons_borough

This function creates a table from the ons_core table. All postcodes in the created table belong to the same borough. To **reduce the request burden (later on) while preserving some representativeness**, I first group each ons_borough dataframe by usertype (0 for residential and 1 for business) and take no more than 30 of each usertype, and then group by LSOAs (neighbourhoods), taking no more than 3 postcodes from each neighbourhood. Then the ONS data for each borough would not be too ridiculous in size.

**Reflections**🤔:
This function removes empty values itself. Last time (mini-project 1) I made mistakes in analysis because I didn't pay enough attention to abnormalities in data.

In [8]:
ons_ealing = ons_borough('ealing', 'E09000009')

C:\Users\Mingyun Wang\AppData\Local\Temp\ipykernel_30660\2009753636.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ons_borough['borough'] = np.where(ons_borough['oslaua'] == code, borough, '')


In [9]:
display(ons_ealing.head(3))
len(ons_ealing)

,oslaua,pcds,msoa11,lsoa11,usertype,lat,long,imd,borough
42190,E09000009,HA0 1AL,E02000239,E01001308,1,51.547777,-0.312886,22868,ealing
42198,E09000009,HA0 1AX,E02000239,E01001308,0,51.548672,-0.312593,22868,ealing
42199,E09000009,HA0 1AY,E02000239,E01001308,0,51.548674,-0.312570,22868,ealing


35

**Reflections**🤔:
I didn't recall np.where() at first, and it wasn't until I asked Claude (I forgot to enter the claude DS105 project though) did I realise I could attach the borough tag instead of making a binary variable and replacing the 'True' values by 'ealing'.

This function seems promising, though. For any London borough henceforth, I will be able to select all active postcodes in it. Then I'd be able to request for data for all postcodes in this borough, and later on I can group data by LSOAs to **analyse at neighbourhoods' average levels**.

### 1.2.2, Requesting Function for Data Collection and Storage
I'm defining here a function that requests for travelling data for all postcodes in a borough. Right below is the API url and the codes loading my API key.

In [10]:
url = 'https://api.tfl.gov.uk/Journey/JourneyResults/{from}/to/{to}'
load_dotenv()
API_KEY = os.getenv('Primary Key')

And below is the **Function for Journey Requests (Data Collection)** 📈:

While requesting data using this function called "***request_journey()***", the commander should **input 4 items**:
1. an "***ons_borough***" pd.DataFrame (as an input collection, format and type specified)
  For each ons_borough table (dataframe), it requests for each postcode in the table:
    - a journey from the postcode to LSE, and
    - a journey from LSE to the postcode
2. a "***borough_name***" string (as a tag for related data subject to this borough)
3. a specific time, AKA variable name "***hour***" as a string in the format 'HHMM' (to specify the planned time of requested journeys)
4. a "***peak***" or "***offpeak***" string (as a tag for related data's time band)

Later on, perhaps in NB02, I will pool journeys from and to LSE together. Specifically, the parameters (duration, cost, etc.) measured for a journey **"BETWEEN"** LSE and a postcode will be the **AVERAGE** of those parameters retrieved from journeys **"FROM LSE"** and from journeys **"TO LSE"**. This is done to compensate for the slight difference between travelling from A to B and from B to A, so that the data can better represent a mean level of travelling Between A and B.

Thus, a raw data json file for a borough will have twice the length compared to the ONS table for that borough.

Request data will be saved in the /data/raw directory as .json files

👇**CHANGE THE INPUT TIME HERE!**

In [16]:
def request_journey(ons_borough, borough_name, hour, peak):
    params = {'date': '20261022', 'time': hour, 'mode': "tube, bus, overground, elizabeth-line, walking, national-rail"}
    ons_borough_data = []    # a place to dump responses
    for pcd in ons_borough['pcds']:
        requesturl = url.replace("{from}", pcd).replace("{to}", "WC2A 2AE")
        response = requests.get(requesturl, headers={"app_key": API_KEY}, timeout=20, params=params)    # travelling from outer to LSE
        ons_borough_data.append(response.json())
        time.sleep(2)
    for pcd in ons_borough['pcds']:
        requesturl = url.replace("{from}", "WC2A 2AE").replace("{to}", pcd)
        response = requests.get(requesturl, headers={"app_key": API_KEY}, timeout=20, params=params)    # travelling from LSE to outer
        ons_borough_data.append(response.json())
        time.sleep(2)
    filename = f'../data/raw/{borough_name}_{peak}_data.json'    # the 'borough_name' variable was added for this
    with open(filename, 'w') as f:    # otherwise in the f string, 'ons_borough' will be a dataframe, which causes errors
        json.dump(ons_borough_data, f, indent=2)

**Reflections**🤔:
I didn't add the 'borough_name' variable in my custom function at first, and inserted directly {ons_borough} into the f-string filename. But since ons_borough is essentially a pd.DataFrame, python inserted the header of the corresponding dataframes into the filename, which caused an error of having too long a filename.

I asked claude and discovered this exact cause of problem. Then I added the 'borough_name' variable to fix it.

Claude also suggested the time.sleep(2) code to avoid overwhelming the TfL API. What a jolly good idea. I'm mind-blown for this.

**Requesting data for travelling between Ealing and LSE during peak hours**
- It takes 6-ish minutes to complete this request
- I'm using 08:00 as the peak sample here

In [17]:
request_journey(ons_ealing, 'ealing', '0800', 'peak')

**Between Ealing and LSE, off-peak hours**
- Also, 6-ish mins to complete this request

In [18]:
request_journey(ons_ealing, 'ealing', '1200', 'offpeak')

## 1.3, EXTENSION: More data from other Boroughs
To represent areas all over the outer London, I've selected another 3 boroughs (other than Ealing) located respectively in the North, East, and South of outer London (Ealing is in the West). More, I've particularly considered those with similar (straight-line) distances to LSE and those connected by London under/overgrounds (one can imagine drawing a circle with LSE at the center and distance from LSE to Ealing Broadway station as the radius). And those borouughs are:

1. Enfield (North), code 	E09000010
2. Barking & Dagenham (East), code E09000002
3. Sutton (South), code E09000029

Note that the requesting cells are not supposed to have obvious output. Go to the 'data/raw' folder to fild the output .json files. Cells featuring ons_borough functions will display the first 3 rows of the selected dataframe; for full selected ons dataframes, also check the 'data/raw' folder.

If this section is folded, **don't click the "22 cells hidden..." button to run those cells!!!**--that will cause errors and unintended outputs!!!

### 1.3.1, Enfield (North), code E09000010

**Selecting ons information from ons_core dataframe**

In [19]:
ons_enfield = ons_borough('enfield', 'E09000010')
display(ons_enfield.head(3))
len(ons_enfield)

C:\Users\Mingyun Wang\AppData\Local\Temp\ipykernel_30660\2009753636.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ons_borough['borough'] = np.where(ons_borough['oslaua'] == code, borough, '')


,oslaua,pcds,msoa11,lsoa11,usertype,lat,long,imd,borough
37744,E09000010,EN1 1AA,E02000291,E01001512,1,51.651916,-0.077106,15781,enfield
37745,E09000010,EN1 1BA,E02000291,E01001406,0,51.641864,-0.068975,11086,enfield
37746,E09000010,EN1 1BB,E02000291,E01001406,0,51.642298,-0.067355,11086,enfield


25

**Peak Hour Data**: 4-ish mins to complete the request
- I'm using 17:00 as the peak hour sample here

In [20]:
request_journey(ons_enfield, 'enfield', '1700', 'peak')

**Off-Peak Hour Data**: 4-ish mins to complete the request

In [30]:
request_journey(ons_enfield, 'enfield', '1200', 'offpeak')

### 1.3.2, Barking & Dagenham (East), code E09000002

**Selecting ONS information**

In [23]:
ons_barking = ons_borough('barking', 'E09000002')
display(ons_barking.head(3))
len(ons_barking)

C:\Users\Mingyun Wang\AppData\Local\Temp\ipykernel_30660\2009753636.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ons_borough['borough'] = np.where(ons_borough['oslaua'] == code, borough, '')


,oslaua,pcds,msoa11,lsoa11,usertype,lat,long,imd,borough
53172,E09000002,IG11 0AB,E02000020,E01000092,0,51.524794,0.124878,6348,barking
53174,E09000002,IG11 0AE,E02000023,E01000096,0,51.531169,0.109388,3997,barking
53175,E09000002,IG11 0AG,E02000023,E01000093,0,51.531241,0.106421,2669,barking


20

**Peak Hour Data**: 2-ish mins to complete this request
- 17:00 used as the sample for peak hours

In [24]:
request_journey(ons_barking, 'barking', '1700', 'peak')

**Off-Peak Hour Data**: 2-ish mins to complete this request

In [25]:
request_journey(ons_barking, 'barking', '1200', 'offpeak')

### 1.3.3, Sutton (South), code E09000029

**Selecting ONS information**

In [26]:
ons_sutton = ons_borough('sutton', 'E09000029')
display(ons_sutton.head(3))
len(ons_sutton)

C:\Users\Mingyun Wang\AppData\Local\Temp\ipykernel_30660\2009753636.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ons_borough['borough'] = np.where(ons_borough['oslaua'] == code, borough, '')


,oslaua,pcds,msoa11,lsoa11,usertype,lat,long,imd,borough
7220,E09000029,CR0 3AQ,E02000850,E01004076,0,51.390100,-0.137450,15853,sutton
7221,E09000029,CR0 3AR,E02000850,E01004076,0,51.389588,-0.140299,15853,sutton
7222,E09000029,CR0 3AS,E02000850,E01004076,0,51.389063,-0.138912,15853,sutton


21

**Peak Hour Data**: 2-ish mins to complete this request
- 08:00 used as the sample for perk hours

In [27]:
request_journey(ons_sutton, 'sutton', '0800', 'peak')

**Off-Peak Hour Data**: 2-ish mins to complete this request

In [28]:
request_journey(ons_sutton, 'sutton', '1200', 'offpeak')

***This is the end of NB01, go to NB02***